### Generate OpenAlex Match DB

In [ ]:
import pandas as pd
import yaml
from tqdm import tqdm
from rapidfuzz import process, fuzz

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
def clean_string(text):
    import re
    if not text:
        return None
    text = text.lower()
    # Remove special characters except letters (a-z), digits, whitespace, and Chinese characters
    text = re.sub(r'[^a-z0-9\s\u4e00-\u9fff]', ' ', text)
    # Replace multiple spaces with a single space and trim leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) > 0:
        return text
    else:
        return None

def clean_author(text):
    import re
    if not text:
        return None
    # Locate "et al" (case insensitive)
    match = re.search(r'\bet\s+al\b', text, flags=re.IGNORECASE)
    if match:
        # Keep only the part before "et al"
        cleaned = text[:match.start()]
        # Remove trailing commas (both ASCII and Chinese '，') and extra whitespace
        cleaned = re.sub(r'[ \s，]+$', ' ', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    else:
        cleaned = text.strip()
    return cleaned

def regex_search_match(input_str):
    import re
    #output = re.search('^CN-(\\d+)(-.*)?', input_str, re.IGNORECASE)
    output = re.search('^(CN-.*)', input_str, re.IGNORECASE)
    return output.group(1).replace('-', '').upper() if output else None

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
openalex_title_year_path = os.environ.get(
    "OPENALEX_TITLE_YEAR_PARQUET",
    os.path.join(dataset_config["path_openalex"], "csv-files_config", "work_title_year.parquet"),
)

In [ ]:
CNROS_patent_paper_TL2023 = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN1_NPL_IncoPat_Formatted.parquet', engine='pyarrow')
CNROS_patent_paper_TL2023

In [ ]:
CNROS_patent_paper_2024_2025 = pd.read_csv(
    dataset_config['path_processed'] + 'CN_CN/CNROS_IncoPat_1format_2024_2025.csv'
).drop(columns=['row_index'])

CNROS_patent_paper_2024_2025['sources'] = 'OSS'
CNROS_patent_paper_2024_2025['paper_title'] = CNROS_patent_paper_2024_2025['paper_title'].astype(str)

for c in ['year', 'first_page', 'last_page']:
    CNROS_patent_paper_2024_2025[c] = (
        CNROS_patent_paper_2024_2025[c]
        .astype('string')
        .str.extract(r'(-?\d+)', expand=False)   # keeps first numeric chunk
        .pipe(pd.to_numeric, errors='coerce')
        .astype(pd.Int16Dtype())
    )

CNROS_patent_paper_2024_2025

In [ ]:
CNROS_patent_paper = pd.concat([CNROS_patent_paper_TL2023, CNROS_patent_paper_2024_2025])
# remove anything after the dot (and the dot itself)
CNROS_patent_paper['apn'] = (
    CNROS_patent_paper['apn']
    .astype('string')
    .str.split('.', n=1).str[0]     # drop .1 etc
    .str.replace(r'^CN', '', regex=True)  # drop leading CN
)
CNROS_patent_paper

In [ ]:
CNROS_patent_paper['paper_title_proc'] = CNROS_patent_paper['paper_title'].parallel_apply(clean_string)
CNROS_patent_paper = CNROS_patent_paper.dropna(subset=['paper_title_proc'])
CNROS_patent_paper

In [ ]:
CN_patent1 = pd.read_stata('./dataset/CN_patents/1985-2025/date.dta')[['申请号', '首次公开号']].rename(columns={'申请号': 'apn', '首次公开号': 'patent_id'})
CN_patent1

In [ ]:
CN_patent2 = pd.read_stata('./dataset/CN_patents/1985-2025/date.dta')[['申请号', '授权公告号']].rename(columns={'申请号': 'apn', '授权公告号': 'patent_id'})
CN_patent2

In [ ]:
CN_patents = pd.concat([CN_patent1, CN_patent2], ignore_index=True)

In [ ]:
CN_patents = CN_patents.drop_duplicates()
CN_patents

In [ ]:
ROS_patent_paper = pd.read_csv(dataset_config['path_pcs'] + '_pcs_oa.csv', usecols=['oaid', 'patent'], engine='pyarrow').rename(columns={'oaid': 'work_id', 'patent': 'patent_id'})
ROS_patent_paper

In [ ]:
ROS_patent_paper['patent_id'] = ROS_patent_paper['patent_id'].parallel_apply(regex_search_match)
ROS_patent_paper = ROS_patent_paper.dropna()
ROS_patent_paper

In [ ]:
oa_patent = CN_patents.merge(ROS_patent_paper, on='patent_id').drop_duplicates()
oa_patent

In [ ]:
oa_patent_pair = oa_patent[['apn', 'patent_id']].drop_duplicates()
oa_patent_pair

In [ ]:
oa_patent_pair.to_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_ROS_apn_patentid.parquet', index=False)

In [ ]:
oa_included = CNROS_patent_paper.merge(oa_patent, on='apn')
oa_included

In [ ]:
oa_works = pd.read_parquet(openalex_title_year_path).rename(columns={'oaid': 'work_id', 'display_name': 'paper_title', 'publication_year': 'year'})
oa_works

In [ ]:
oa_works_merged = oa_included.merge(oa_works, on='work_id')
oa_works_merged

In [ ]:
d_result = {'apn': [], 'work_id': [], 'confidence': [], 'incopat_title': [], 'paper_title': [], 'year': []}
idx_no_match = []

for work_id, sub_df in tqdm(oa_works_merged.groupby(by='work_id')):
    paper_title = sub_df['paper_title_y'].iloc[0]
    paper_year = sub_df['year_y'].iloc[0]
    oa_title = clean_string(paper_title)
    match = process.extractOne(oa_title, sub_df.paper_title_proc, scorer=fuzz.WRatio)

    if match and match[1] > 90:
        df_matches = sub_df[sub_df.paper_title_proc == match[0]]
        incopat_title = df_matches.iloc[0].paper_title_x
        match_apns = df_matches.apn

        for apn in match_apns:
            d_result['apn'].append(apn)
            d_result['work_id'].append(work_id)
            d_result['confidence'].append(match[1])
            d_result['incopat_title'].append(incopat_title)
            d_result['paper_title'].append(paper_title)
            d_result['year'].append(paper_year)

        no_match = sub_df[sub_df.paper_title_proc != match[0]].index
        idx_no_match += no_match.to_list()
    else:
        idx_no_match += sub_df.index.to_list()

In [ ]:
valid_matches = pd.DataFrame(d_result).drop_duplicates(subset=['apn', 'work_id'])
valid_matches

In [ ]:
valid_matches[['apn', 'work_id', 'paper_title', 'year']].to_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_IncoPat_OA_intersect.parquet', index=False)

In [ ]:
incopat_raw = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN1_NPL_IncoPat_Formatted.parquet', engine='pyarrow').rename(columns={'paper_title': 'incopat_title'})
incopat_raw

In [ ]:
result_no_matches = incopat_raw.merge(valid_matches[['apn', 'work_id', 'incopat_title']], on=['apn', 'incopat_title'], how='left').drop_duplicates(subset=['apn', 'incopat_title', 'authors', 'journal_name', 'year'])
result_no_matches = result_no_matches[result_no_matches.work_id.isna()].rename(columns={'incopat_title': 'paper_title'})
result_no_matches

In [ ]:
result_no_matches[['apn', 'paper_title', 'authors', 'journal_name', 'year']].to_parquet(dataset_config['path_processed'] + 'CN_CN/CN2_IncoPat_only.parquet', index=False)